# 04 Stage 2: classification

Forecasts, for each district and issue day t, whether a damaging flood, landslide, heavy-rainfall
or lightning incident is recorded on days t+1 to t+3 (objective 2), and tests whether the Stage 1
regions help (objective 3).

Protocol (fixed before any test-season number was looked at):

- Tune each classifier on leave-one-season-out folds inside 2017 to 2022 (mean PR-AUC).
- Fit the chosen setting on 2017 to 2022, calibrate the score on 2023 with isotonic regression.
- Score 2024 to 2025 once; then the 2026 check (issue days to 16 August).
- The "best model" is the one with the highest mean fold PR-AUC in tuning, not the best test score.
- Ranking metrics (PR-AUC, ROC-AUC, POD, FAR and CSI at a 5% alert budget, a share of the scored
  test rows) use the raw score; Brier and the Brier skill score use the calibrated probability.
- 95% intervals: weekly block bootstrap, 1,000 resamples, paired differences against climatology.

Feature sets: A (known at t; ERA5 analysis of day t, an analysis-based hindcast), B (A plus
observed ERA5 rain for t+1 to t+3, a perfect-forecast ceiling) and NOWX (no weather at all, the
floor). Everything is seeded with `config.SEED`.

- **Inputs:** `data/processed/feature-matrix.parquet` (notebook 01) and `district-regions.csv` (notebook 03)
- **Outputs:** `results/test-*.csv`, `tuning-*.csv`, `best-params.csv`, `skill-per-season.csv`,
  `check-2026-set-a.csv`, `calibration-sensitivity.csv`; `data/processed/predictions.parquet`; the
  fitted models in `data/processed/models/stage2-models.joblib` (not tracked in git; notebook 05 loads them)
- **Runtime:** about 16 minutes (leave-one-season-out tuning and 1,000-resample bootstraps)

## Setup

In [1]:
import json
import sys
import time
from pathlib import Path

# Make `src` importable whether the notebook runs from notebooks/ or the repository root.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import joblib
import numpy as np
import pandas as pd

from src import config as C
from src import evaluate as E
from src import experiments as X
from src import features as Fe
from src import models as M
from src import sensitivity as S

T_START = time.time()
F = pd.read_parquet(C.PROCESSED / "feature-matrix.parquet")
enc = pd.read_csv(C.PROCESSED / "district-regions.csv", index_col="district")
data = X.attach_regions(F, enc)
tr, va, te, ck = data.train, data.validate, data.test, data.check
print(
    f"rows: train {len(tr):,}, validate {len(va):,}, test {len(te):,}, check {len(ck):,}; seed {C.SEED}"
)
print(
    "positive rate in percent:",
    {
        k: round(100 * d["y"].mean(), 2)
        for k, d in zip(("train", "validate", "test", "check"), (tr, va, te, ck))
    },
)
assert round(100 * te["y"].mean(), 2) == 8.46

clim_fit = E.fit_climatology(F, C.TRAIN_SEASONS)
clim = {s: E.climatology_score(data.block(s), clim_fit) for s in ("validate", "test", "check")}
y_te, dates_te = te["y"].to_numpy(), te["date"]
N_BOOT = 1000
results = {}  # every experiment's named (score, prob) pairs on the test rows

rows: train 97,482, validate 16,247, test 32,494, check 10,626; seed 42
positive rate in percent: {'train': np.float64(6.65), 'validate': np.float64(4.44), 'test': np.float64(8.46), 'check': np.float64(5.78)}


## 1. Baselines

Climatology (district x month rate, 2017 to 2022), persistence (a damaging incident dated t-3 to t
already entered by day t), the Dahal and Hasegawa (2008) thresholds applied to ERA5 (144 mm on day
t or 182 mm over 72 h), a refitted ERA5 threshold (the 3-day rain percentile cut with the best CSI
on 2017 to 2022) and a year-only control (logistic regression on year and month). The control has
no district term, so it sits below climatology by construction; it only shows whether year alone ranks district-days. No
model uses year as a feature. Notebook 05 (section 5) adds year to a district-history control.

In [2]:
cut = E.fit_rain_threshold(F, C.TRAIN_SEASONS)
thr_score, thr_alert = E.rain_threshold_score(te, cut)
dahal = E.dahal_threshold_score(te)
pers = E.persistence_score(te)
yo_prob = S.year_only_score(F)

baselines = {
    "climatology": (clim["test"], clim["test"]),
    "persistence": (pers, pers),
    "dahal_threshold": (dahal, dahal),
    "refit_threshold": (thr_score, thr_score),
    "year_only_control": (yo_prob, yo_prob),
}
base_tab = X.summarise(baselines, y_te, clim["test"], dates_te, n_boot=N_BOOT)
# Rule baselines fire on their own alerts, not on a 5% budget: report their actual POD, FAR and CSI too.
for name, flag in (
    ("persistence", pers),
    ("dahal_threshold", dahal),
    ("refit_threshold", thr_alert),
):
    own = E.alert_metrics(y_te, flag, alert=flag.astype(bool))
    for k, v in own.items():
        base_tab.loc[name, f"own_{k}"] = v
base_tab.to_csv(C.RESULTS / "test-baselines.csv")
results.update(baselines)
print(
    f"refitted threshold: 3-day rain percentile >= {cut}; Dahal thresholds fire on {100 * dahal.mean():.2f}% of test rows"
)
print(X.fmt(base_tab).to_string())
print(base_tab[[c for c in base_tab if c.startswith("own_")]].dropna().round(3).to_string())

refitted threshold: 3-day rain percentile >= 0.79; Dahal thresholds fire on 1.41% of test rows
                                 pr_auc                      bss                   csi    pod    far
model                                                                                               
climatology        0.143 [0.120, 0.166]     0.000 [0.000, 0.000]  0.078 [0.064, 0.091]  0.115  0.805
persistence        0.101 [0.082, 0.121]  -0.795 [-0.943, -0.664]  0.068 [0.060, 0.084]  0.102  0.828
dahal_threshold    0.091 [0.072, 0.112]  -0.207 [-0.267, -0.154]  0.050 [0.035, 0.070]  0.076  0.871
refit_threshold    0.150 [0.118, 0.182]  -2.972 [-3.588, -2.424]  0.083 [0.064, 0.102]  0.122  0.794
year_only_control  0.128 [0.100, 0.160]   -0.006 [-0.014, 0.002]  0.069 [0.039, 0.097]  0.103  0.826
                 own_pod  own_far  own_csi  own_alert_rate
model                                                     
persistence        0.172    0.820    0.096           0.081
dahal_threshold    0.

## 2. Leave-one-season-out tuning on 2017 to 2022 (feature set A)

Small grids (appendix table): the aim is a fair comparison of the four families, not the last
decimal of a search. The RBF SVM trains on a stratified subsample of 12,000 rows per fold.

In [3]:
seasons_tr = tr["season"].to_numpy()
Xa, ya = X.xy(tr, Fe.FEATURES_A, "y")
best, folds, oof = {}, [], {}
for name in ("lr", "rf", "hgb", "svm_linear", "svm_rbf"):
    t0 = time.time()
    best[name], table, oof[name] = M.loso_tune(name, Xa, ya, seasons_tr)
    folds.append(table)
    print(f"{name}: best {best[name]} in {time.time() - t0:.0f} s")
folds = pd.concat(folds, ignore_index=True)
folds.to_csv(C.RESULTS / "tuning-folds.csv", index=False)
summary = (
    folds.groupby(
        ["model"]
        + [c for c in folds if c not in ("model", "held_out_season", "pr_auc", "seconds")],
        dropna=False,
    )["pr_auc"]
    .agg(["mean", "std"])
    .round(4)
)
summary.to_csv(C.RESULTS / "tuning-summary.csv")
tune_best = {
    n: folds[(folds["model"] == n) & np.all([folds[k] == v for k, v in best[n].items()], axis=0)][
        "pr_auc"
    ].mean()
    for n in best
}
BEST = max(tune_best, key=tune_best.get)
pd.DataFrame(
    [
        {"model": n, "params": json.dumps(best[n]), "loso_mean_pr_auc": round(tune_best[n], 4)}
        for n in best
    ]
).to_csv(C.RESULTS / "best-params.csv", index=False)
print("mean fold PR-AUC of the chosen settings:", {n: round(v, 4) for n, v in tune_best.items()})
print(f"best model by tuning: {BEST} ({M.NAMES[BEST]})")

  lr {'C': 0.01}: mean fold PR-AUC 0.1363


  lr {'C': 0.1}: mean fold PR-AUC 0.1363


  lr {'C': 1.0}: mean fold PR-AUC 0.1363
lr: best {'C': 0.01} in 1 s


  rf {'min_samples_leaf': 20, 'max_features': 'sqrt'}: mean fold PR-AUC 0.1499


  rf {'min_samples_leaf': 20, 'max_features': 0.5}: mean fold PR-AUC 0.1474


  rf {'min_samples_leaf': 100, 'max_features': 'sqrt'}: mean fold PR-AUC 0.1460


... (6 outputs left out)

  svm_linear {'C': 0.01}: mean fold PR-AUC 0.1350


  svm_linear {'C': 0.1}: mean fold PR-AUC 0.1350
svm_linear: best {'C': 0.1} in 4 s


  svm_rbf {'C': 0.3, 'gamma': 'scale'}: mean fold PR-AUC 0.1175


  svm_rbf {'C': 1.0, 'gamma': 'scale'}: mean fold PR-AUC 0.1069
svm_rbf: best {'C': 0.3, 'gamma': 'scale'} in 60 s
mean fold PR-AUC of the chosen settings: {'lr': np.float64(0.1363), 'rf': np.float64(0.1499), 'hgb': np.float64(0.1475), 'svm_linear': np.float64(0.135), 'svm_rbf': np.float64(0.1175)}
best model by tuning: rf (Random forest)


## 3. Main result: four classifiers on feature set A, test 2024 to 2025

In [4]:
fits = {}
for name in best:
    fits[(name, "A")] = X.run(data, name, best[name], Fe.FEATURES_A)
    print(f"{name} set A fitted and calibrated in {fits[(name, 'A')]['seconds']:.0f} s")
main_named = {
    "climatology": baselines["climatology"],
    **{name: (fits[(name, "A")]["score_test"], fits[(name, "A")]["prob_test"]) for name in best},
}
main_tab = X.summarise(main_named, y_te, clim["test"], dates_te, n_boot=N_BOOT)
main_tab["feature_set"] = "A"
main_tab.to_csv(C.RESULTS / "test-main-set-a.csv")
results.update({f"{n}_A": v for n, v in main_named.items() if n != "climatology"})
print(X.fmt(main_tab).to_string())

lr set A fitted and calibrated in 0 s


rf set A fitted and calibrated in 5 s


hgb set A fitted and calibrated in 3 s


svm_linear set A fitted and calibrated in 0 s


svm_rbf set A fitted and calibrated in 5 s


                           pr_auc                     bss                   csi    pod    far
model                                                                                        
climatology  0.143 [0.120, 0.166]    0.000 [0.000, 0.000]  0.078 [0.064, 0.091]  0.115  0.805
lr           0.177 [0.141, 0.216]   0.005 [-0.007, 0.017]  0.098 [0.078, 0.115]  0.142  0.760
rf           0.165 [0.130, 0.202]  -0.006 [-0.018, 0.007]  0.081 [0.062, 0.103]  0.119  0.799
hgb          0.175 [0.139, 0.211]  -0.006 [-0.019, 0.008]  0.090 [0.069, 0.113]  0.131  0.778
svm_linear   0.177 [0.141, 0.215]  -0.015 [-0.040, 0.005]  0.097 [0.077, 0.114]  0.141  0.762
svm_rbf      0.153 [0.125, 0.186]  -0.010 [-0.023, 0.004]  0.082 [0.063, 0.101]  0.120  0.796


## 4. Feature-set ablation: no weather (floor), set A, set B (perfect forecast ceiling)

Same tuned settings as set A (no retuning), so the comparison isolates the information added.

In [5]:
abl_named = {"climatology": baselines["climatology"]}
for fs in ("NOWX", "B"):
    for name in best:
        fits[(name, fs)] = X.run(data, name, best[name], Fe.FEATURE_SETS[fs])
        abl_named[f"{name}_{fs}"] = (fits[(name, fs)]["score_test"], fits[(name, fs)]["prob_test"])
for name in best:
    abl_named[f"{name}_A"] = main_named[name]
fs_tab = X.summarise(abl_named, y_te, clim["test"], dates_te, n_boot=N_BOOT)
fs_tab.index.name = "run"
fs_tab["model"] = [
    k.rsplit("_", 1)[0] if k != "climatology" else "climatology" for k in fs_tab.index
]
fs_tab["feature_set"] = [k.rsplit("_", 1)[1] if k != "climatology" else "" for k in fs_tab.index]
fs_tab.to_csv(C.RESULTS / "test-feature-sets.csv")
results.update({k: v for k, v in abl_named.items() if k != "climatology"})
view = (
    fs_tab.reset_index(drop=True)
    .pivot(index="model", columns="feature_set", values="pr_auc")
    .round(4)
)
print("PR-AUC by model and feature set:\n", view.to_string())
print("\nbest model:\n", X.fmt(fs_tab.loc[[f"{BEST}_NOWX", f"{BEST}_A", f"{BEST}_B"]]).to_string())

PR-AUC by model and feature set:
 feature_set               A       B    NOWX
model                                      
climatology  0.1425     NaN     NaN     NaN
hgb             NaN  0.1748  0.2201  0.1641
lr              NaN  0.1774  0.2122  0.1638
rf              NaN  0.1652  0.2193  0.1431
svm_linear      NaN  0.1769  0.2113  0.1641
svm_rbf         NaN  0.1527  0.1787  0.1407

best model:
                        pr_auc                      bss                   csi    pod    far
run                                                                                       
rf_NOWX  0.143 [0.117, 0.172]  -0.017 [-0.027, -0.005]  0.070 [0.054, 0.085]  0.104  0.825
rf_A     0.165 [0.130, 0.202]   -0.006 [-0.018, 0.007]  0.081 [0.062, 0.103]  0.119  0.799
rf_B     0.219 [0.156, 0.299]     0.028 [0.002, 0.059]  0.121 [0.080, 0.170]  0.172  0.710


## 5. Do the regions help? Four location encodings (objective 3)

Set A already carries the district x month history rates, so "none" is not location-free; the
question is whether the Stage 1 regions add anything beyond that history. Encodings: the K-means
label (one-hot), the GMM posterior probabilities, the distances to the K-means centroids, and a
plain district one-hot. Run for the best model and for logistic regression, where the literature
(Piernik and Morzy 2021) expects cluster features to matter most.

In [6]:
reg_named = {"climatology": baselines["climatology"]}
reg_rows = []
for name in sorted({BEST, "lr"}):
    for enc_name, extra in data.region_cols.items():
        key = f"{name}_{enc_name}"
        if enc_name == "none":
            reg_named[key] = main_named[name]
        else:
            r = X.run(data, name, best[name], Fe.FEATURES_A + extra)
            fits[(name, f"A+{enc_name}")] = r
            reg_named[key] = (r["score_test"], r["prob_test"])
reg_tab = X.summarise(reg_named, y_te, clim["test"], dates_te, n_boot=N_BOOT)
reg_tab.index.name = "run"
reg_tab["model"] = [
    k.rsplit("_", 1)[0] if k != "climatology" else "climatology" for k in reg_tab.index
]
reg_tab["encoding"] = [k.rsplit("_", 1)[1] if k != "climatology" else "" for k in reg_tab.index]
# Paired difference against the same model without regions (bootstrap on the same weeks).
for name in sorted({BEST, "lr"}):
    pair = {
        f"{name}_none": reg_named[f"{name}_none"],
        **{f"{name}_{e}": reg_named[f"{name}_{e}"] for e in data.region_cols if e != "none"},
    }
    d = E.block_bootstrap(y_te, pair, clim["test"], dates_te, n_boot=N_BOOT)
    for _, row in d[d["metric"] == "pr_auc"].iterrows():
        reg_tab.loc[
            row["model"],
            ["pr_auc_diff_vs_none", "pr_auc_diff_vs_none_lo", "pr_auc_diff_vs_none_hi"],
        ] = [row["diff_vs_ref"], row["diff_lo"], row["diff_hi"]]
reg_tab.to_csv(C.RESULTS / "test-region-encodings.csv")
results.update({k: v for k, v in reg_named.items() if k != "climatology"})
print(
    reg_tab[
        [
            "model",
            "encoding",
            "pr_auc",
            "pr_auc_diff_vs_none",
            "pr_auc_diff_vs_none_lo",
            "pr_auc_diff_vs_none_hi",
            "bss",
            "csi",
        ]
    ]
    .round(4)
    .to_string()
)

                    model   encoding  pr_auc  pr_auc_diff_vs_none  pr_auc_diff_vs_none_lo  pr_auc_diff_vs_none_hi     bss     csi
run                                                                                                                              
climatology   climatology             0.1425                  NaN                     NaN                     NaN  0.0000  0.0781
lr_none                lr       none  0.1774               0.0000                  0.0000                  0.0000  0.0047  0.0979
lr_label               lr      label  0.1738              -0.0036                 -0.0081                  0.0005  0.0027  0.0919
lr_posterior           lr  posterior  0.1758              -0.0016                 -0.0061                  0.0022  0.0038  0.0954
lr_distance            lr   distance  0.1776               0.0002                 -0.0053                  0.0051  0.0044  0.0960
lr_district            lr   district  0.1748              -0.0026                 -0.0091 

## 6. Sensitivity: the rain-driven target (flood, landslide, heavy rainfall; no lightning)

Lightning is 36% of the positives and peaks before the monsoon, so the pooled target mixes two
regimes. The same pipeline is run on `y_rain` with sets A and B for the best model and logistic
regression, against its own climatology.

In [7]:
clim_rain_fit = {
    "cell": tr.groupby(["district", "month"])["y_rain"].mean(),
    "month": tr.groupby("month")["y_rain"].mean(),
}
clim_rain = E.climatology_score(te, clim_rain_fit)
y_rain_te = te["y_rain"].to_numpy()
rain_named = {"climatology": (clim_rain, clim_rain)}
for name in sorted({BEST, "lr"}):
    for fs in ("A", "B"):
        r = X.run(data, name, best[name], Fe.FEATURE_SETS[fs], target="y_rain")
        fits[(name, fs, "y_rain")] = r
        rain_named[f"{name}_{fs}"] = (r["score_test"], r["prob_test"])
rain_tab = X.summarise(rain_named, y_rain_te, clim_rain, dates_te, n_boot=N_BOOT)
rain_tab.index.name = "run"
rain_tab["target"] = "y_rain"
rain_tab.to_csv(C.RESULTS / "test-rain-target.csv")
print(f"rain-driven target, base rate {100 * y_rain_te.mean():.2f} percent")
print(X.fmt(rain_tab).to_string())

rain-driven target, base rate 5.49 percent
                           pr_auc                    bss                   csi    pod    far
run                                                                                         
climatology  0.123 [0.092, 0.153]   0.000 [0.000, 0.000]  0.084 [0.065, 0.111]  0.148  0.838
lr_A         0.177 [0.133, 0.224]  0.016 [-0.001, 0.033]  0.122 [0.095, 0.147]  0.208  0.772
lr_B         0.229 [0.153, 0.318]   0.050 [0.014, 0.091]  0.163 [0.110, 0.218]  0.267  0.706
rf_A         0.163 [0.122, 0.210]  0.005 [-0.012, 0.022]  0.104 [0.078, 0.136]  0.180  0.802
rf_B         0.231 [0.141, 0.340]   0.044 [0.004, 0.093]  0.163 [0.100, 0.233]  0.267  0.706


## 7. Skill per season, including the 2026 check

Base rates differ between seasons (4.4% in 2023, 9.0% in 2024, 7.9% in 2025, 5.8% in 2026), so
PR-AUC is reported beside the season's base rate and the climatology PR-AUC. 2023 is the
calibration season, so its Brier skill is in-sample for the calibrator and is marked as such.

In [8]:
rows = []
for name in best:
    r = fits[(name, "A")]
    for split in ("validate", "test", "check"):
        d = data.block(split)
        for s in sorted(d["season"].unique()):
            m = d["season"].to_numpy() == s
            met = E.metrics(
                d["y"].to_numpy()[m], r[f"score_{split}"][m], clim[split][m], r[f"prob_{split}"][m]
            )
            cl = E.metrics(d["y"].to_numpy()[m], clim[split][m], clim[split][m])
            rows.append(
                {
                    "model": name,
                    "season": s,
                    "role": {
                        "validate": "calibration (in-sample)",
                        "test": "test",
                        "check": "2026 check",
                    }[split],
                    "base_rate": met["base_rate"],
                    "clim_pr_auc": cl["pr_auc"],
                    **{k: met[k] for k in ("pr_auc", "roc_auc", "bss", "pod", "far", "csi")},
                }
            )
season_tab = pd.DataFrame(rows)
season_tab.to_csv(C.RESULTS / "skill-per-season.csv", index=False)
print(season_tab[season_tab["model"] == BEST].round(3).to_string(index=False))
# 2026 check with bootstrap intervals for the best model and climatology.
chk_named = {
    "climatology": (clim["check"], clim["check"]),
    **{n: (fits[(n, "A")]["score_check"], fits[(n, "A")]["prob_check"]) for n in best},
}
chk_tab = X.summarise(chk_named, ck["y"].to_numpy(), clim["check"], ck["date"], n_boot=N_BOOT)
chk_tab.to_csv(C.RESULTS / "check-2026-set-a.csv")
print("\n2026 check:\n", X.fmt(chk_tab).to_string())

model  season                    role  base_rate  clim_pr_auc  pr_auc  roc_auc    bss   pod   far   csi
   rf    2023 calibration (in-sample)      0.044        0.081   0.086    0.690  0.034 0.120 0.893 0.060
   rf    2024                    test      0.090        0.158   0.202    0.731 -0.008 0.136 0.755 0.096
   rf    2025                    test      0.079        0.131   0.125    0.662 -0.004 0.092 0.855 0.060
   rf    2026              2026 check      0.058        0.087   0.100    0.673  0.027 0.099 0.885 0.056



2026 check:
                            pr_auc                    bss                   csi    pod    far
model                                                                                       
climatology  0.087 [0.061, 0.119]   0.000 [0.000, 0.000]  0.048 [0.027, 0.066]  0.085  0.902
lr           0.102 [0.074, 0.129]  0.025 [-0.002, 0.055]  0.066 [0.047, 0.086]  0.116  0.866
rf           0.100 [0.075, 0.126]   0.027 [0.004, 0.057]  0.056 [0.038, 0.074]  0.099  0.885
hgb          0.100 [0.076, 0.127]   0.023 [0.000, 0.053]  0.060 [0.045, 0.083]  0.106  0.878
svm_linear   0.102 [0.074, 0.130]  0.011 [-0.017, 0.044]  0.061 [0.045, 0.085]  0.107  0.876
svm_rbf      0.108 [0.079, 0.138]  0.023 [-0.002, 0.054]  0.082 [0.058, 0.103]  0.142  0.836


## 8. Calibration sensitivity: 2023 only versus pooled out-of-fold 2017 to 2023

2023 has the lowest positive rate, so a calibrator fitted on it alone will under-predict 2024 to
2025. The alternative pools the leave-one-season-out scores of 2017 to 2022 with the 2023 scores.

In [9]:
rows = []
for name in best:
    r = fits[(name, "A")]
    m = r["model"]
    p23 = r["prob_test"]
    pooled_scores = np.concatenate([oof[name], r["score_validate"]])
    pooled_y = np.concatenate([ya, va["y"].to_numpy()])
    m_pooled = M.Calibrated(name, best[name])
    m_pooled.model = m.model
    m_pooled.refit_calibration(pooled_scores, pooled_y)
    p_pool = m_pooled.predict_proba(te[Fe.FEATURES_A].to_numpy(float))
    for lab, p in (("calibrated on 2023", p23), ("pooled OOF 2017 to 2023", p_pool)):
        met = E.metrics(y_te, r["score_test"], clim["test"], p)
        rows.append(
            {
                "model": name,
                "calibration": lab,
                "brier": met["brier"],
                "bss": met["bss"],
                "mean_prob": p.mean(),
                "base_rate": y_te.mean(),
            }
        )
    if name == BEST:
        results[f"{BEST}_A_pooledcal"] = (r["score_test"], p_pool)
cal_tab = pd.DataFrame(rows)
cal_tab.to_csv(C.RESULTS / "calibration-sensitivity.csv", index=False)
print(cal_tab.round(4).to_string(index=False))

     model             calibration  brier     bss  mean_prob  base_rate
        lr      calibrated on 2023 0.0756  0.0047     0.0501     0.0846
        lr pooled OOF 2017 to 2023 0.0746  0.0182     0.0666     0.0846
        rf      calibrated on 2023 0.0765 -0.0063     0.0467     0.0846
        rf pooled OOF 2017 to 2023 0.0748  0.0163     0.0675     0.0846
       hgb      calibrated on 2023 0.0765 -0.0061     0.0485     0.0846
       hgb pooled OOF 2017 to 2023 0.0745  0.0198     0.0677     0.0846
svm_linear      calibrated on 2023 0.0772 -0.0151     0.0517     0.0846
svm_linear pooled OOF 2017 to 2023 0.0746  0.0180     0.0664     0.0846
   svm_rbf      calibrated on 2023 0.0768 -0.0102     0.0477     0.0846
   svm_rbf pooled OOF 2017 to 2023 0.0752  0.0104     0.0671     0.0846


## 9. Save predictions and models for notebook 05

In [10]:
pred = pd.concat([te, ck])[
    [
        "district",
        "date",
        "season",
        "split",
        "month",
        "region",
        "belt",
        "y",
        "y_rain",
        "y_flood",
        "y_landslide",
        "y_heavy_rain",
        "y_lightning",
    ]
].reset_index(drop=True)
pred["clim"] = np.concatenate([clim["test"], clim["check"]])
for name in best:
    for fs in ("NOWX", "A", "B"):
        r = fits[(name, fs)]
        pred[f"score_{name}_{fs}"] = np.concatenate([r["score_test"], r["score_check"]])
        pred[f"prob_{name}_{fs}"] = np.concatenate([r["prob_test"], r["prob_check"]])
pred[f"prob_{BEST}_A_pooledcal"] = np.concatenate(
    [results[f"{BEST}_A_pooledcal"][1], np.full(len(ck), np.nan)]
)
pred.to_parquet(C.PROCESSED / "predictions.parquet", index=False)
(C.PROCESSED / "models").mkdir(exist_ok=True)
joblib.dump(
    {
        "best": BEST,
        "params": best,
        "features_A": Fe.FEATURES_A,
        "model_A": fits[(BEST, "A")]["model"],
        "model_B": fits[(BEST, "B")]["model"],
        "lr_A": fits[("lr", "A")]["model"],
    },
    C.PROCESSED / "models" / "stage2-models.joblib",
)
print(pred.shape, "-> data/processed/predictions.parquet")
print(f"notebook run time: {(time.time() - T_START) / 60:.1f} min")

(43120, 45) -> data/processed/predictions.parquet
notebook run time: 16.2 min
